In [1]:
# CELL 1: IMPORTS AND PATH SETUP
import sys
sys.path.append("src")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import quantum_algorithms as qa

In [2]:
# CELL 2: GLOBAL PLOT STYLE

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "legend.fontsize": 11,
})

In [3]:
# CELL 3: EXPERIMENT CONFIGURATION

EXPERIMENT_NAME = "Deutsch-Jozsa Rotation Noise Study"
RANDOM_SEED = 12345

# Number of input qubits: n = 1 to 9
n_values = range(1, 10)

# Rotation angle from 0 to pi, equivalent to 0 to 180 degrees
theta_values = np.linspace(0, np.pi, 181)

# Shots kept for reporting/comparison, but exact statevector probabilities are used
shots = 1024

# Choose target-qubit strategy:
# "middle" = one target qubit per n
# "all_input" = all input qubits 0 to n-1
TARGET_MODE = "middle"

axes = {
    "X": (1, 0, 0),
    "Y": (0, 1, 0),
    "Z": (0, 0, 1),
}

functions = {
    "constant_0": qa.deutsch_jozsa.f_constant_0,
    "constant_1": qa.deutsch_jozsa.f_constant_1,
    "balanced_parity": qa.deutsch_jozsa.f_balanced_parity,
}

error_positions = {
    "E1_before_H": qa.deutsch_jozsa.deutsch_jozsa_error1,
    "E2_after_first_H": qa.deutsch_jozsa.deutsch_jozsa_error2,
    "E3_after_oracle": qa.deutsch_jozsa.deutsch_jozsa_error3,
    "E4_after_final_H": qa.deutsch_jozsa.deutsch_jozsa_error4,
}

label_map = {
    "no_error": "No Error",
    "E1_before_H": r"$\mathcal{E}_1$ before first $H$",
    "E2_after_first_H": r"$\mathcal{E}_2$ after first $H$",
    "E3_after_oracle": r"$\mathcal{E}_3$ after oracle",
    "E4_after_final_H": r"$\mathcal{E}_4$ after final $H$",
}

error_order = [
    "no_error",
    "E1_before_H",
    "E2_after_first_H",
    "E3_after_oracle",
    "E4_after_final_H",
]

error_only_order = [
    "E1_before_H",
    "E2_after_first_H",
    "E3_after_oracle",
    "E4_after_final_H",
]

In [4]:
# CELL 4: HELPER FUNCTIONS


def get_target_qubits(n, mode="middle"):
    """
    Select target qubits for the rotation error.
    
    mode = "middle":
        Use only the middle input qubit.
    
    mode = "all_input":
        Use all input qubits from 0 to n-1.
    """

    if mode == "middle":
        return [n // 2]

    if mode == "all_input":
        return list(range(n))

    raise ValueError(f"Unknown target mode: {mode}")


def get_P0_exact(state, n):
    """
    Exact probability of measuring the all-zero input register.

    This uses the final statevector directly instead of shot sampling.
    """

    probs = qa.deutsch_jozsa.measure_probs_first_n(state, n)
    return float(np.clip(probs[0], 0.0, 1.0))


def get_success_probability(function_name, P0):
    """
    Deutsch-Jozsa success probability.

    For constant functions:
        success = P(input register = 0...0)

    For balanced functions:
        success = P(input register != 0...0) = 1 - P0
    """

    if function_name in ["constant_0", "constant_1"]:
        return P0

    if function_name == "balanced_parity":
        return 1 - P0

    raise ValueError(f"Unknown function name: {function_name}")


def standard_error_from_probability(p, shots):
    """
    Binomial standard error.

    np.clip prevents tiny floating-point errors such as
    p = 1.0000000000000002 from causing sqrt warnings.
    """

    p = np.clip(p, 0.0, 1.0)
    return np.sqrt(p * (1 - p) / shots)


def make_result_row(
    n,
    theta,
    theta_deg,
    axis_name,
    target_qubit,
    function_name,
    error_position,
    P0,
    shots
):
    """
    Create one standardized result row.
    """

    success = get_success_probability(function_name, P0)

    return {
        "n": n,
        "theta_rad": theta,
        "theta_deg": theta_deg,
        "axis": axis_name,
        "target_qubit": target_qubit,
        "function": function_name,
        "error_position": error_position,
        "P0": P0,
        "success_probability": success,
        "std_error_if_sampled": standard_error_from_probability(success, shots),
        "shots_reference": shots,
    }

In [ ]:
# ============================================================
# CELL 5: RUN FULL SIMULATION
# ============================================================

results = []

start_time = time.time()

for n in n_values:
    print(f"Running n = {n}")

    target_qubits = get_target_qubits(n, TARGET_MODE)

    for function_name, f in functions.items():

        # Compute no-error reference once for each n and function
        state_ref = qa.deutsch_jozsa.deutsch_jozsa(n, f)
        P0_ref = get_P0_exact(state_ref, n)

        for target_qubit in target_qubits:

            for theta in theta_values:
                theta_deg = np.degrees(theta)

                for axis_name, axis in axes.items():

                    # Store no-error baseline for each theta and axis
                    results.append(
                        make_result_row(
                            n=n,
                            theta=theta,
                            theta_deg=theta_deg,
                            axis_name=axis_name,
                            target_qubit=target_qubit,
                            function_name=function_name,
                            error_position="no_error",
                            P0=P0_ref,
                            shots=shots,
                        )
                    )

                    # Rotation-error cases E1, E2, E3, E4
                    for error_name, error_function in error_positions.items():

                        state_error = error_function(
                            n,
                            f,
                            theta,
                            target_qubit,
                            axis,
                        )

                        P0_error = get_P0_exact(state_error, n)

                        results.append(
                            make_result_row(
                                n=n,
                                theta=theta,
                                theta_deg=theta_deg,
                                axis_name=axis_name,
                                target_qubit=target_qubit,
                                function_name=function_name,
                                error_position=error_name,
                                P0=P0_error,
                                shots=shots,
                            )
                        )

end_time = time.time()

df = pd.DataFrame(results)

print("Simulation complete.")
print("Dataset shape:", df.shape)
print(f"Runtime: {(end_time - start_time) / 60:.2f} minutes")

df.head()

Running n = 1
Running n = 2


In [ ]:
# ============================================================
# CELL 6: SAVE RAW DATASET
# ============================================================

raw_csv_name = "dja_rotation_error_exact_results_n1_to_9.csv"

df.to_csv(raw_csv_name, index=False)

print(f"Raw results saved as: {raw_csv_name}")

In [ ]:
# ============================================================
# CELL 7: AVERAGE SUCCESS OVER FUNCTIONS
# ============================================================

df_avg = (
    df.groupby(
        [
            "n",
            "theta_rad",
            "theta_deg",
            "axis",
            "target_qubit",
            "error_position",
        ],
        as_index=False
    )
    .agg(
        average_success=("success_probability", "mean"),
        average_std_error_if_sampled=("std_error_if_sampled", "mean"),
        shots_reference=("shots_reference", "first"),
    )
)

avg_csv_name = "dja_average_success_exact_results_n1_to_9.csv"

df_avg.to_csv(avg_csv_name, index=False)

print(f"Average success results saved as: {avg_csv_name}")
df_avg.head()

In [ ]:
# ============================================================
# CELL 8: ERROR SENSITIVITY RANKING
# ============================================================

ranking_df = (
    df_avg[df_avg["error_position"] != "no_error"]
    .groupby("error_position", as_index=False)
    .agg(
        mean_average_success=("average_success", "mean"),
        min_average_success=("average_success", "min"),
        max_average_success=("average_success", "max"),
    )
    .sort_values("mean_average_success")
)

ranking_df["mean_success_loss"] = 1 - ranking_df["mean_average_success"]

ranking_df["rank"] = range(1, len(ranking_df) + 1)

ranking_df["interpretation"] = ranking_df["rank"].apply(
    lambda r: "Most damaging error position" if r == 1
    else "Least damaging error position" if r == len(ranking_df)
    else "Intermediate sensitivity"
)

ranking_csv_name = "dja_error_sensitivity_ranking_n1_to_9.csv"

ranking_df.to_csv(ranking_csv_name, index=False)

print(f"Error sensitivity ranking saved as: {ranking_csv_name}")
ranking_df

In [ ]:
# ============================================================
# CELL 9: COMPARATIVE TABLE BY n AND ERROR POSITION
# ============================================================

comparative_table = df_avg.pivot_table(
    values="average_success",
    index="n",
    columns="error_position",
    aggfunc="mean"
)

comparative_table = comparative_table.round(4)

comparative_table["worst_error_success"] = (
    comparative_table[
        [
            "E1_before_H",
            "E2_after_first_H",
            "E3_after_oracle",
            "E4_after_final_H",
        ]
    ]
    .min(axis=1)
)

comparative_table["success_loss_vs_no_error"] = (
    comparative_table["no_error"]
    - comparative_table["worst_error_success"]
)

comparative_csv_name = "dja_comparative_table_by_n_n1_to_9.csv"

comparative_table.to_csv(comparative_csv_name)

print(f"Comparative table saved as: {comparative_csv_name}")

comparative_table

In [ ]:
# ============================================================
# CELL 10: PLOT FUNCTION 1
# SUCCESS PROBABILITY FOR ONE FUNCTION
# ============================================================

def plot_error_positions(df, n_plot, axis_plot, target_plot, function_plot):
    """
    Plot success probability for one Boolean function and compare all error positions.
    """

    plot_df = df[
        (df["n"] == n_plot) &
        (df["axis"] == axis_plot) &
        (df["target_qubit"] == target_plot) &
        (df["function"] == function_plot)
    ].sort_values("theta_deg")

    if plot_df.empty:
        print("No data found for this plot.")
        return

    plt.figure(figsize=(9, 6))

    for error_pos in error_order:
        subset = plot_df[
            plot_df["error_position"] == error_pos
        ].sort_values("theta_deg")

        if subset.empty:
            continue

        plt.plot(
            subset["theta_deg"],
            subset["success_probability"],
            linewidth=2,
            label=label_map[error_pos],
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Success probability")
    plt.title(
        f"Deutsch-Jozsa success under rotation errors\n"
        f"function={function_plot}, n={n_plot}, axis={axis_plot}, target={target_plot}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Error position")
    plt.tight_layout()

    filename_pdf = f"plot_error_positions_n{n_plot}_{axis_plot}_q{target_plot}_{function_plot}.pdf"
    filename_png = f"plot_error_positions_n{n_plot}_{axis_plot}_q{target_plot}_{function_plot}.png"

    plt.savefig(filename_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(filename_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figures: {filename_pdf}, {filename_png}")

In [ ]:
def plot_average_success(df_avg, n_plot, axis_plot, target_plot):
    """
    Plot average success probability over constant_0, constant_1,
    and balanced_parity.
    """

    plot_df = df_avg[
        (df_avg["n"] == n_plot) &
        (df_avg["axis"] == axis_plot) &
        (df_avg["target_qubit"] == target_plot)
    ].sort_values("theta_deg")

    if plot_df.empty:
        print("No data found for this plot.")
        return

    plt.figure(figsize=(10, 6))

    for error_pos in error_order:
        subset = plot_df[
            plot_df["error_position"] == error_pos
        ].sort_values("theta_deg")

        if subset.empty:
            continue

        plt.plot(
            subset["theta_deg"],
            subset["average_success"],
            linewidth=2,
            label=label_map[error_pos],
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Average success probability")
    plt.title(
        f"Average Deutsch-Jozsa success under rotation errors\n"
        f"n={n_plot}, axis={axis_plot}, target qubit={target_plot}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Error position")
    plt.tight_layout()

    filename_pdf = f"plot_average_success_n{n_plot}_{axis_plot}_q{target_plot}.pdf"
    filename_png = f"plot_average_success_n{n_plot}_{axis_plot}_q{target_plot}.png"

    plt.savefig(filename_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(filename_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figures: {filename_pdf}, {filename_png}")

In [ ]:
# ============================================================
# CELL 12: PLOT FUNCTION 3
# ERROR IMPACT / SUCCESS LOSS
# ============================================================

def plot_error_impact(df_avg, n_plot, axis_plot, target_plot):
    """
    Plot loss in average success probability.

    success_loss =
        no-error average success
        - error average success
    """

    plot_df = df_avg[
        (df_avg["n"] == n_plot) &
        (df_avg["axis"] == axis_plot) &
        (df_avg["target_qubit"] == target_plot)
    ].sort_values("theta_deg")

    if plot_df.empty:
        print("No data found for this plot.")
        return

    no_error = (
        plot_df[plot_df["error_position"] == "no_error"]
        [["theta_deg", "average_success"]]
        .rename(columns={"average_success": "success_no_error"})
        .sort_values("theta_deg")
    )

    plt.figure(figsize=(10, 6))

    for error_pos in error_only_order:

        subset = (
            plot_df[plot_df["error_position"] == error_pos]
            [["theta_deg", "average_success"]]
            .sort_values("theta_deg")
        )

        if subset.empty:
            continue

        merged = pd.merge(
            subset,
            no_error,
            on="theta_deg",
            how="inner"
        )

        merged["success_loss"] = (
            merged["success_no_error"]
            - merged["average_success"]
        )

        plt.plot(
            merged["theta_deg"],
            merged["success_loss"],
            linewidth=2,
            label=label_map[error_pos],
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Loss in average success probability")

    plt.title(
        f"Error impact on Deutsch-Jozsa success\n"
        f"n={n_plot}, axis={axis_plot}, target qubit={target_plot}"
    )

    plt.xlim(0, 180)

    # Loss should never exceed 1
    plt.ylim(-0.02, 1.02)

    plt.grid(True, alpha=0.3)
    plt.legend(title="Error position")
    plt.tight_layout()

    filename_pdf = (
        f"plot_error_impact_n{n_plot}_{axis_plot}_q{target_plot}.pdf"
    )

    filename_png = (
        f"plot_error_impact_n{n_plot}_{axis_plot}_q{target_plot}.png"
    )

    plt.savefig(filename_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(filename_png, dpi=300, bbox_inches="tight")

    plt.show()

    print(f"Saved figures: {filename_pdf}, {filename_png}")

In [ ]:
# ============================================================
# CELL 13: PLOT FUNCTION 4
# AXIS COMPARISON
# ============================================================

def plot_axis_comparison(df_avg, n_plot, target_plot, error_position_plot):
    """
    Compare X, Y, and Z rotation axes for one error position.
    """

    plot_df = df_avg[
        (df_avg["n"] == n_plot) &
        (df_avg["target_qubit"] == target_plot) &
        (df_avg["error_position"] == error_position_plot)
    ].sort_values("theta_deg")

    if plot_df.empty:
        print("No data found for this plot.")
        return

    plt.figure(figsize=(10, 6))

    for axis_name in ["X", "Y", "Z"]:
        subset = plot_df[
            plot_df["axis"] == axis_name
        ].sort_values("theta_deg")

        if subset.empty:
            continue

        plt.plot(
            subset["theta_deg"],
            subset["average_success"],
            linewidth=2,
            label=f"{axis_name}-axis rotation",
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Average success probability")

    plt.title(
        f"Comparison of rotation axes\n"
        f"n={n_plot}, target qubit={target_plot}, error={label_map[error_position_plot]}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Rotation axis")
    plt.tight_layout()

    filename_pdf = f"plot_axis_comparison_n{n_plot}_{error_position_plot}_q{target_plot}.pdf"
    filename_png = f"plot_axis_comparison_n{n_plot}_{error_position_plot}_q{target_plot}.png"

    plt.savefig(filename_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(filename_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figures: {filename_pdf}, {filename_png}")

In [ ]:
def plot_n_comparison(df_avg, axis_plot, error_position_plot):
    """
    Compare average success probability for different input sizes n.
    """

    plot_df = df_avg[
        (df_avg["axis"] == axis_plot) &
        (df_avg["error_position"] == error_position_plot)
    ].sort_values(["n", "theta_deg"])

    if plot_df.empty:
        print("No data found for this plot.")
        return

    plt.figure(figsize=(10, 6))

    for n in sorted(plot_df["n"].unique()):
        subset = plot_df[plot_df["n"] == n]

        subset_avg = (
            subset.groupby("theta_deg", as_index=False)
            .agg(average_success=("average_success", "mean"))
            .sort_values("theta_deg")
        )

        if subset_avg.empty:
            continue

        plt.plot(
            subset_avg["theta_deg"],
            subset_avg["average_success"],
            linewidth=2,
            label=f"n={n}",
        )

    plt.xlabel(r"Rotation angle $\theta$ (degrees)")
    plt.ylabel("Average success probability")
    plt.title(
        f"Scaling with number of input qubits\n"
        f"axis={axis_plot}, error={label_map[error_position_plot]}"
    )

    plt.xlim(0, 180)
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Input qubits", ncol=2)
    plt.tight_layout()

    filename_pdf = f"plot_n_comparison_{axis_plot}_{error_position_plot}.pdf"
    filename_png = f"plot_n_comparison_{axis_plot}_{error_position_plot}.png"

    plt.savefig(filename_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(filename_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figures: {filename_pdf}, {filename_png}")

In [ ]:
# ============================================================
# CELL 15
# ============================================================

def plot_scalability_at_fixed_angle(
    df_avg,
    theta_choice=90,
    axis_choice="X",
    error_choice="E4_after_final_H"
):
    """
    Plot average success probability versus n for a fixed angle,
    axis, and error position.
    """

    scale_df = df_avg[
        (df_avg["theta_deg"] == theta_choice) &
        (df_avg["axis"] == axis_choice) &
        (df_avg["error_position"] == error_choice)
    ]

    if scale_df.empty:
        print("No data found for this plot.")
        return

    grouped = (
        scale_df.groupby("n", as_index=False)
        .agg(average_success=("average_success", "mean"))
        .sort_values("n")
    )

    plt.figure(figsize=(8, 5))

    plt.plot(
        grouped["n"],
        grouped["average_success"],
        marker="o",
        linewidth=2,
    )

    plt.xlabel("Number of input qubits n")
    plt.ylabel("Average success probability")
    plt.title(
        f"Scalability under {label_map[error_choice]} noise\n"
        f"axis={axis_choice}, θ={theta_choice}°"
    )

    plt.xticks(sorted(grouped["n"].unique()))
    plt.ylim(0, 1.05)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    filename_pdf = f"plot_scalability_{error_choice}_{axis_choice}_{theta_choice}deg.pdf"
    filename_png = f"plot_scalability_{error_choice}_{axis_choice}_{theta_choice}deg.png"

    plt.savefig(filename_pdf, dpi=300, bbox_inches="tight")
    plt.savefig(filename_png, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Saved figures: {filename_pdf}, {filename_png}")

In [ ]:
# ============================================================
# CELL 16: RUN SELECTED PLOTS
# ============================================================

# 1. Average success curves (small, medium, large n)
plot_average_success(df_avg, n_plot=1, axis_plot="X", target_plot=0)
plot_average_success(df_avg, n_plot=5, axis_plot="X", target_plot=2)
plot_average_success(df_avg, n_plot=9, axis_plot="X", target_plot=4)

# 2. Function-specific behaviour
plot_error_positions(
    df,
    n_plot=5,
    axis_plot="X",
    target_plot=2,
    function_plot="constant_1",
)

plot_error_positions(
    df,
    n_plot=5,
    axis_plot="X",
    target_plot=2,
    function_plot="balanced_parity",
)

# 3. Error impact
plot_error_impact(
    df_avg,
    n_plot=5,
    axis_plot="Y",
    target_plot=2,
)

# 4. Axis comparison for all error positions
for error_pos in [
    "E1_before_H",
    "E2_after_first_H",
    "E3_after_oracle",
    "E4_after_final_H",
]:
    plot_axis_comparison(
        df_avg,
        n_plot=5,
        target_plot=2,
        error_position_plot=error_pos,
    )

# 5. Scaling with n for X, Y, Z
for axis in ["X", "Y", "Z"]:

    plot_n_comparison(
        df_avg,
        axis_plot=axis,
        error_position_plot="E1_before_H",
    )

    plot_n_comparison(
        df_avg,
        axis_plot=axis,
        error_position_plot="E4_after_final_H",
    )

# 6. Fixed-angle scalability
for axis in ["X", "Y", "Z"]:

    plot_scalability_at_fixed_angle(
        df_avg,
        theta_choice=90,
        axis_choice=axis,
        error_choice="E4_after_final_H",
    )

print("\nError Sensitivity Ranking")
display(ranking_df)

print("\nComparative Table")
display(comparative_table)